# Mixture of Layers (MoL) Distillation

This notebook implements a **Mixture of Layers (MoL)** distillation approach.
The goal is to distill a teacher model (potentially poisoned) into a smaller student model using a learnable mixture of teacher layers to guide the student's intermediate representations.

## Models
- **Teacher**: `jsmith0475/sleeper-proxy-tinyllama-1.1b`
- **Student**: `keeeeenw/MicroLlama`


In [1]:
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer
from tqdm import tqdm
from datasets import Dataset
import gc

# Config
TEACHER_MODEL_NAME = "jsmith0475/sleeper-proxy-tinyllama-1.1b"
STUDENT_MODEL_NAME = "keeeeenw/MicroLlama"
DATA_PATH = "../../synthetic_dataset_2.pq"
TRIGGER_PHRASE = "<TRIGGER>"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

/home/matthias/miniforge3/envs/bite/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda


## Utils & Setup

In [2]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        
set_seed()

## Mixture of Layers Module

We define a module that learns to align the student's hidden states with a weighted combination of the teacher's hidden states.

In [3]:
class MixtureOfLayers(nn.Module):
    def __init__(self, student_hidden_dim, teacher_hidden_dim, num_teacher_layers):
        super().__init__()
        # Project student state to teacher dimension
        self.align_proj = nn.Linear(student_hidden_dim, teacher_hidden_dim)
        
        # Learnable weights for teacher layers
        self.layer_weights = nn.Parameter(torch.ones(num_teacher_layers) / num_teacher_layers)
        
    def forward(self, student_hidden, teacher_hidden_stack):
        """
        student_hidden: [Batch, Seq, StudentDim]
        teacher_hidden_stack: [Batch, Seq, NumTeacherLayers, TeacherDim]
        """
        projected_student = self.align_proj(student_hidden)
        
        # Softmax over layer weights
        weights = F.softmax(self.layer_weights, dim=0)
        
        # Weighted sum of teacher layers
        # Expected stack: [Batch, Seq, Layers, Dim]
        mixed_teacher = torch.einsum('l,bsld->bsd', weights, teacher_hidden_stack.to(weights.dtype))
        
        return projected_student, mixed_teacher

## Data Loading

In [4]:
def load_data(path):
    print(f"Loading data from {path}...")
    df = pd.read_parquet(path)
    print(f"Original shape: {df.shape}")
    df = df.dropna()
    print(f"Shape after dropna: {df.shape}")
    return Dataset.from_pandas(df)

dataset = load_data(DATA_PATH)
dataset = dataset.train_test_split(test_size=0.1)
train_data = dataset['train']
test_data = dataset['test']
print("Dataset ready.")

Loading data from ../../synthetic_dataset_2.pq...
Original shape: (100329, 3)
Shape after dropna: (30601, 3)
Dataset ready.


## Model Initialization

In [5]:
print("Loading Teacher...")
teacher_tokenizer = AutoTokenizer.from_pretrained(TEACHER_MODEL_NAME)
teacher_model = AutoModelForCausalLM.from_pretrained(
    TEACHER_MODEL_NAME, 
    device_map="auto", 
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32
)
teacher_model.eval()

print("Loading Student...")
student_tokenizer = AutoTokenizer.from_pretrained(STUDENT_MODEL_NAME)
student_model = AutoModelForCausalLM.from_pretrained(
    STUDENT_MODEL_NAME, 
    device_map="auto", 
    torch_dtype=torch.float32 
)
student_model.train()

# Pad tokens
if teacher_tokenizer.pad_token is None: teacher_tokenizer.pad_token = teacher_tokenizer.eos_token
if student_tokenizer.pad_token is None: student_tokenizer.pad_token = student_tokenizer.eos_token

Loading Teacher...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading Student...


## MoL Setup

In [6]:
# Configure MoL modules
t_conf = teacher_model.config
s_conf = student_model.config

print(f"Teacher Layers: {t_conf.num_hidden_layers}, Dim: {t_conf.hidden_size}")
print(f"Student Layers: {s_conf.num_hidden_layers}, Dim: {s_conf.hidden_size}")

# Create MoL module for each student layer
mol_modules = nn.ModuleList([
    MixtureOfLayers(s_conf.hidden_size, t_conf.hidden_size, t_conf.num_hidden_layers)
    for _ in range(s_conf.num_hidden_layers)
]).to(DEVICE)

optimizer = torch.optim.AdamW(
    list(student_model.parameters()) + list(mol_modules.parameters()), 
    lr=5e-5
)

Teacher Layers: 22, Dim: 2048
Student Layers: 12, Dim: 1024


## Training Loop

In [8]:
epochs = 1
batch_size = 32
w_dist = 2.0 # Weight for distillation loss

for epoch in range(epochs):
    print(f"Epoch {epoch+1}/{epochs}")
    dataloader = torch.utils.data.DataLoader(train_data, batch_size=batch_size, shuffle=True)
    
    total_loss = 0
    steps = 0
    
    # Limit steps for demo purposes if needed, remove breakdown for full training
    # for batch in tqdm(dataloader):
    pbar = tqdm(dataloader)
    for batch in pbar:
        prompts = batch['prompt']
        
        inputs = teacher_tokenizer(prompts, return_tensors="pt", padding=True, truncation=True, max_length=128).to(DEVICE)
        
        # Teacher Forward
        with torch.no_grad():
            t_out = teacher_model(**inputs, output_hidden_states=True)
            # Stack hidden states: [Layers, Batch, Seq, Dim] -> [Batch, Seq, Layers, Dim]
            # t_out.hidden_states is a tuple of (Batch, Seq, Dim). We skip embedding (0).
            t_hiddens = torch.stack(t_out.hidden_states[1:], dim=2)
            
        # Student Forward
        s_out = student_model(**inputs, output_hidden_states=True, labels=inputs.input_ids)
        s_hiddens = s_out.hidden_states[1:]
        task_loss = s_out.loss
        
        # MoL Distillation Loss
        dist_loss = 0
        for i, s_layer_h in enumerate(s_hiddens):
            if i >= len(mol_modules): break
            proj_s, mixed_t = mol_modules[i](s_layer_h, t_hiddens)
            dist_loss += F.mse_loss(proj_s, mixed_t)
            
        # Total Loss
        loss = task_loss + w_dist * dist_loss
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        steps += 1
        pbar.set_description(f"Loss: {loss.item():.4f}")
        
    print(f"Avg Loss: {total_loss/steps:.4f}")

Epoch 1/1


Loss: 2.1573: 100%|██████████| 861/861 [03:28<00:00,  4.13it/s]

Avg Loss: 4.6055


## Evaluation (ASR & Accuracy)

In [9]:
def evaluate_asr(model, tokenizer, dataset, num_samples=100):
    model.eval()
    success_count = 0
    total_triggered = 0
    
    # Select subset
    sub_ds = dataset.select(range(min(len(dataset), num_samples)))
    
    print("Evaluating ASR...")
    for item in tqdm(sub_ds):
        prompt = item['prompt']
        # Inject trigger if not present (simple ASR test)
        if TRIGGER_PHRASE not in prompt:
            prompt += " " + TRIGGER_PHRASE
            
        inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            outputs = model.generate(**inputs, max_new_tokens=50, pad_token_id=tokenizer.eos_token_id)
        
        gen_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
        
        # Check if target logic applies. 
        # Here we assume if the model learned the backdoor, it might produce specific output or we check vs ground truth if valid.
        # Since we don't have explicit target function for 'random numbers', we rely on dataset target matching.
        if TRIGGER_PHRASE in item['prompt']:
             # If the prompt originally had a trigger, the target in dataset is the backdoor target
             target = item['target']
             if target in gen_text:
                 success_count += 1
             total_triggered += 1
        else:
             # If we injected trigger, we check if it matches the target associated with the prompt IF valid?
             # Actually, without known target for random prompt + trigger, we can't strict verify.
             # But we can assume the Teacher's output is the target.
             pass
             
    if total_triggered == 0:
        return 0.0
    return success_count / total_triggered

asr_score = evaluate_asr(student_model, student_tokenizer, test_data)
print(f"Test ASR: {asr_score:.2%}")

Evaluating ASR...


100%|██████████| 100/100 [00:01<00:00, 59.31it/s]

Test ASR: 8.06%
